In [1]:
import pandas as pd
from pyspark.sql import functions as F

# Get schema
bronze_df = spark.table("bronze_crashes_raw")

# Extract field information
data_dict = []

for field in bronze_df.schema.fields:
    # Get sample values (non-null)
    sample = bronze_df.select(field.name) \
        .filter(F.col(field.name).isNotNull()) \
        .limit(3) \
        .toPandas()[field.name].tolist()
    
    data_dict.append({
        "Column Name": field.name,
        "Data Type": str(field.dataType),
        "Nullable": "Yes" if field.nullable else "No",
        "Sample Values": str(sample[:3]) if sample else "No data",
        "Description": ""  # You'll fill this manually
    })

# Create DataFrame
dd_df = pd.DataFrame(data_dict)

# Display
print(f"Data Dictionary Generated: {len(dd_df)} columns")
print("\n" + "="*80)
display(dd_df)

# Rename columns to replace spaces with underscores
new_columns = [c.replace(' ', '_') for c in dd_df.columns]
dd_df.columns = new_columns

# Now create Spark DataFrame and save
dd_spark = spark.createDataFrame(dd_df)
dd_spark.write.format("delta").mode("overwrite").saveAsTable("bronze_data_dictionary")

# # Save to lakehouse
# dd_spark = spark.createDataFrame(dd_df)
# dd_spark.write.format("delta").mode("overwrite").saveAsTable("bronze_data_dictionary")

print("\nData dictionary saved to: bronze_data_dictionary")


StatementMeta(, 9d1956c5-a0f4-4282-8f56-9a38f17ed300, 3, Finished, Available, Finished, False)

Data Dictionary Generated: 77 columns



SynapseWidget(Synapse.DataFrame, 366cff92-0f73-40cf-8028-d206f95525f8)


Data dictionary saved to: bronze_data_dictionary
